In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append("../../..")

import analytiq_data as ad
import logging
logging.basicConfig(level=logging.INFO)
# Textract often emits KEY_VALUE_SET blocks without CHILD; textractor logs INFO per block (not an error).
ad.aws.textract.configure_textractor_logging()
logger = logging.getLogger("test_textract")

In [ ]:
ad.common.setup()

In [ ]:
analytiq_client = ad.common.get_analytiq_client(env="test")
aws_client = await ad.aws.get_aws_client_async(analytiq_client)

In [ ]:
document_id = "67aa2cae086d8b1d7826ef36"
document_id = "679c9abbfcb627f388ffd59f"
document_id = "69bee8d779aab97f8eac6aef"
doc = await ad.common.get_doc(analytiq_client, document_id)
if doc is None:
    raise Exception("Document not found")

file_name = doc.get("pdf_file_name") or doc.get("mongo_file_name")
file_obj = await ad.common.get_file_async(analytiq_client, 
                                          file_name)
if file_obj is None:
    raise Exception("File not found")

blob = file_obj["blob"]

In [ ]:
logger.info(f"blob size: {len(blob)}")
logger.info(f"blob head: {blob[:100]}")

In [ ]:

blocks = await ad.aws.textract.run_textract(analytiq_client, 
                                            blob, 
                                            feature_types=["LAYOUT", "TABLES", "FORMS", "SIGNATURES"],
                                            document_id=document_id)
logger.info(f"blocks: {len(blocks)}")

In [ ]:
blocks

In [ ]:
import textractor

In [ ]:
document = textractor.parsers.response_parser.parse(blocks)

In [ ]:
# Save the document to a markdown file
with open("document.md", "w") as f:
    f.write(document.to_markdown())

In [ ]:
document.to_markdown()